In [ ]:
python -m data.extract_dataset --cat "Baby Products"

In [ ]:
python -m vocab.mine_tokens --cat "Baby Products" --min-freq 30


In [ ]:
python -m vocab.extend_vocabulary --cat "Baby products" --tokenizer-out-dir "vocab/tokenizer/v1"
python -m vocab.validate_tokenizer --cat "Baby Products" --tokenizer-dir "vocab/tokenizer/v1"

In [ ]:
python -m embedding_warmup.train_new_embeddings --config embedding_warmup/configs/warmup_config.yaml --cat "Baby Products"

In [ ]:
"""Warm up new-token embeddings via MLM with all base model parameters frozen.

Only the rows in the embedding matrix corresponding to newly added vocab tokens
receive gradient updates (enforced via a backward hook).

Usage:
    python -m embedding_warmup.train_new_embeddings --config embedding_warmup/configs/warmup_config.yaml
    python -m embedding_warmup.train_new_embeddings --config embedding_warmup/configs/warmup_config.yaml --cat "Baby Products"
"""

import argparse
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import torch
import yaml
from datasets import Dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, Trainer, TrainingArguments

from utils.training_funcs import (
    LossLoggingCallback,
    NewTokenEvalCallback,
    NewTokenForceMaskCollator,
    build_input_text,
    load_new_token_ids,
    register_new_token_grad_mask,
    select_callback_dataset,
)


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

@dataclass
class WarmupConfig:
    cat: str
    tokenizer_dir: Path = field(default_factory=lambda: Path("vocab/tokenizer/v1"))
    new_tokens_dir: Path = field(default_factory=lambda: Path("vocab/new_tokens"))
    data_dir: Path = field(default_factory=lambda: Path("data/raw"))
    processed_dir: Path = field(default_factory=lambda: Path("data/processed"))
    checkpoint_dir: Path = field(default_factory=lambda: Path("models/embedding_warmup_checkpoint"))
    loss_log_file: Path = field(default_factory=lambda: Path("logs/embedding_warmup_loss.jsonl"))
    eval_log_file: Path = field(default_factory=lambda: Path("logs/embedding_warmup_eval.jsonl"))
    train_split: float = 0.9
    random_seed: int = 42
    callback_set_size: int = 10_000
    max_length: int = 512
    mlm_probability: float = 0.15
    per_device_train_batch_size: int = 8
    eval_steps: int = 64
    save_steps: int = 1000
    num_train_epochs: int = 3
    learning_rate: float = 5e-5
    bf16: bool = True
    fp16: bool = False
    dataloader_pin_memory: bool = False


_PATH_FIELDS = {
    "tokenizer_dir", "new_tokens_dir", "data_dir", "processed_dir",
    "checkpoint_dir", "loss_log_file", "eval_log_file",
}


def load_config(path: Path, overrides: dict | None = None) -> WarmupConfig:
    with open(path) as f:
        data = yaml.safe_load(f)
    if overrides:
        data.update({k: v for k, v in overrides.items() if v is not None})
    for key in _PATH_FIELDS:
        if key in data:
            data[key] = Path(data[key])
    return WarmupConfig(**data)


# ---------------------------------------------------------------------------
# Model setup
# ---------------------------------------------------------------------------

def _freeze_base_params(model) -> None:
    """Freeze everything; unfreeze only the token embedding weight."""
    for param in model.parameters():
        param.requires_grad_(False)
    model.model.embeddings.tok_embeddings.weight.requires_grad_(True)


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def _load_and_split(
    cfg: WarmupConfig,
    tokenizer: AutoTokenizer,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    data_path = cfg.data_dir / f"data_processed_{cfg.cat}.csv"
    print(f"[warmup] loading data from {data_path}")
    df = pd.read_csv(data_path, low_memory=False)
    print(f"[warmup] {len(df):,} rows")

    for col in ("TITLE", "OVERVIEW", "BULLETS"):
        df[col] = df[col].fillna("").astype(str)

    df["_input_text"] = df.apply(
        lambda r: build_input_text(r, tokenizer.sep_token), axis=1
    )

    train_df = df.sample(frac=cfg.train_split, random_state=cfg.random_seed)
    test_df = df.drop(train_df.index).reset_index(drop=True)
    train_df = train_df.reset_index(drop=True)

    cfg.processed_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(cfg.processed_dir / f"train_{cfg.cat}.csv", index=False)
    test_df.to_csv(cfg.processed_dir / f"test_{cfg.cat}.csv", index=False)
    print(f"[warmup] train={len(train_df):,}  test={len(test_df):,}  -> {cfg.processed_dir}")

    return train_df, test_df


def _make_hf_dataset(
    df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    max_length: int,
) -> Dataset:
    def _tokenize(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        return {k: enc[k] for k in ("input_ids", "attention_mask") if k in enc}

    ds = Dataset.from_dict({"text": df["_input_text"].tolist()})
    return ds.map(_tokenize, batched=True, remove_columns=["text"])


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------


/Users/agrimaggarwal/Projects/ecom-item-embeddings/ecom-item-embeddings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

config = "embedding_warmup/configs/warmup_config.yaml"
cat = 'Baby Products'
cfg = load_config(config, overrides={"cat": cat})

print(f"[warmup] cat={cfg.cat!r}  tokenizer_dir={cfg.tokenizer_dir}")

tokenizer = AutoTokenizer.from_pretrained(str(cfg.tokenizer_dir))
model = AutoModelForMaskedLM.from_pretrained(str(cfg.tokenizer_dir))
model.resize_token_embeddings(len(tokenizer))

new_token_ids = load_new_token_ids(tokenizer, cfg.new_tokens_dir, cfg.cat)

_freeze_base_params(model)
register_new_token_grad_mask(
    model.model.embeddings.tok_embeddings.weight,
    new_token_ids,
)
print(f"[warmup] base params frozen; grad mask on {len(new_token_ids)} new token rows")

train_df, test_df = _load_and_split(cfg, tokenizer)

callback_df = select_callback_dataset(test_df, tokenizer, new_token_ids, cfg.callback_set_size)
callback_df.to_csv(cfg.processed_dir / f"callback_{cfg.cat}.csv", index=False)
print(f"[warmup] callback set: {len(callback_df):,} samples")

train_ds = _make_hf_dataset(train_df, tokenizer, cfg.max_length)
callback_ds = _make_hf_dataset(callback_df, tokenizer, cfg.max_length)

collator = NewTokenForceMaskCollator(
    tokenizer=tokenizer,
    new_token_ids=new_token_ids,
    mlm_probability=cfg.mlm_probability,
)

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"[warmup] device={device}")

for p in (cfg.loss_log_file.parent, cfg.eval_log_file.parent, cfg.checkpoint_dir):
    p.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(cfg.checkpoint_dir),
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    learning_rate=cfg.learning_rate,
    save_steps=cfg.save_steps,
    save_total_limit=5,
    logging_steps=1,
    logging_strategy="steps",
    report_to="none",
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)


TypeError: WarmupConfig.__init__() got an unexpected keyword argument 'bf16'

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator,
    callbacks=[
        LossLoggingCallback(log_file=cfg.loss_log_file),
        NewTokenEvalCallback(
            model=model,
            tokenizer=tokenizer,
            callback_dataset=callback_ds,
            new_token_ids=new_token_ids,
            log_file=cfg.eval_log_file,
            collator=collator,
            eval_steps=cfg.eval_steps,
            device=device,
        ),
    ],
)

print("[warmup] starting training")
trainer.train()
print("[warmup] done")


In [11]:
### HI
import json
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from torch import Tensor
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
    AutoModelForMaskedLM
)


In [58]:
import ast
text = """[" LOOKING FOR GIFTS FOR MOM? This is a great mom gift box for any occasion. Great mothers day gifts for mom. Birthday gifts for mom from daughters, sons, kids and husbands. Customers' top choice when searching for mothers day gift ideas, gift basket for mom, mother-in-law, wife, gifts for mom who has everything unique. It's the best mom gift set for ANYTIME you want to show your appreciation for a trying mother!"," WOW HER WITH A CUSTOMIZED MOM GIFT BOX. When she opens it, she will know she's the best mom in the world. Bursting with scent, this mom gift basket includes a 12oz coffee mug with lid and spoon, cozy sock, lavender candle, ocean&rose bath bomb, love knot necklace, jewelry dish, compact mirror, and greeting card. No assembly required. Safe and nice package which can be sent as a Thanksgiving/Christmas/Valentines/Anniversary gift directly."," NECKLACE WITH INSPIRTIONAL WORDS. What better way to show appreciation and forever love to mom than having a beautiful love knot necklace with a card saying: \"Whenever you feel overwhelmed, remember whose mother you are. To the world you are a mother, but to our family you are the world\". As a mother, reading those words gives her such strength. It made her feel special and loved. This necklace is destined to become her favorite fashion accessory and guaranteed to win over the attention!"," PRACTICAL MOM GIFTS. Mom can refresh herself and enjoy her coffee time using this best mom ever mug. Whenever she uses it, she is reminded of the fact that you think she's awesome. Cozy cotton socks made with anti-slippery nature of letters warm her feet and prevent her from sliding. Creative words on the bottom of the socks also make her joyful."," FUNNY MOM CANDLE & PETAL BATH BOMB. \"Mom' s Last Nerve, Oh look...it's on " humorous quote can make mom smile every time she looks at them. She will enjoy sleep/yoga/spa/bath time with 100% natural soy wax candle which burns smokelessly & evenly for up to 50 hours. Every time mom pamper herself with anything from this gift set, she'll think about who gave her such a thoughtful present. It's not just a nice gift, but a reminder that someone cares enough to get her something so nice."]"""
# json.loads(text)
ast.literal_eval(text)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)

In [60]:
data_path

'data/processed/test_Baby Products.csv'

In [72]:
def build_input_text(row: dict[str, Any], sep_token: str) -> str:

    title = str(row.get("TITLE", "") or "")
    overview = '\n'.join([f"{k} : {v}" for k,v in json.loads(row.get("OVERVIEW", "")).items()]) # str( or "")
    bullets = '\n'.join([f'- {x}' for x in json.loads(row.get("BULLETS", ""))]) #str(row.get("BULLETS", "") or "")

    return f"TITLE: {title} {sep_token}\nATTRIBUTES:\n{overview} {sep_token}\nFEATURES:\n{bullets}"

data_path = 'data/raw/data_Baby Products.csv'
#  B0BYRL4PMR
df = pd.read_csv(data_path, low_memory=False)
df["_input_text"] = df.apply(
        lambda r: build_input_text(r, "[SEP]"), axis=1
    )

In [86]:
# df[df['ASIN']=='B0BYRL4PMR']

import re

text = df.loc[4332, 'BULLETS']
text_2 = text[:30] + text[-30:]
print(text_2)
clean_text = re.sub(r'[a-zA-Z0-9\\]*\\u[0-9a-fA-F]{4}[a-zA-Z0-9\\]*', '', text)
clean_text_2 = re.sub(r'[a-zA-Z0-9\\]*\\u[0-9a-fA-F]{4}[a-zA-Z0-9\\]*', '', text_2)

["\uD83C\uDF81 LOOKING FOR GIFo get her something so nice."]


In [120]:
[print(x) for x in json.loads(text)]

🎁 LOOKING FOR GIFTS FOR MOM? This is a great mom gift box for any occasion. Great mothers day gifts for mom. Birthday gifts for mom from daughters, sons, kids and husbands. Customers' top choice when searching for mothers day gift ideas, gift basket for mom, mother-in-law, wife, gifts for mom who has everything unique. It's the best mom gift set for ANYTIME you want to show your appreciation for a trying mother!
🎁 WOW HER WITH A CUSTOMIZED MOM GIFT BOX. When she opens it, she will know she's the best mom in the world. Bursting with scent, this mom gift basket includes a 12oz coffee mug with lid and spoon, cozy sock, lavender candle, ocean&rose bath bomb, love knot necklace, jewelry dish, compact mirror, and greeting card. No assembly required. Safe and nice package which can be sent as a Thanksgiving/Christmas/Valentines/Anniversary gift directly.
🎁 NECKLACE WITH INSPIRTIONAL WORDS. What better way to show appreciation and forever love to mom than having a beautiful love knot necklace 

[None, None, None, None, None]

In [117]:
print(text[1761:])

"\uD83C\uDF81 FUNNY MOM CANDLE & PETAL BATH BOMB. \"Mom' s Last Nerve, Oh look...it's on fire\uD83D\uDD25\" humorous quote can make mom smile every time she looks at them. She will enjoy sleep/yoga/spa/bath time with 100% natural soy wax candle which burns smokelessly & evenly for up to 50 hours. Every time mom pamper herself with anything from this gift set, she'll think about who gave her such a thoughtful present. It's not just a nice gift, but a reminder that someone cares enough to get her something so nice."]


In [93]:
# text[:20]
# text_2 = text[:30] + text[396:440] + text[-30:]
# clean_text_2 = re.sub(r'[a-zA-Z0-9\\]*\\u[0-9a-fA-F]{4}[a-zA-Z0-9\\]*', '', text_2)
text_2 = [json.dumps(x) for x in json.loads(text)]
clean_text_2 = [re.sub(r'[a-zA-Z0-9\\]*\\u[0-9a-fA-F]{4}[a-zA-Z0-9\\]*', '', x) for x in text_2]
# text.find('ppreciation for a trying')

In [134]:
text_2[4][90:100]

'ire\\ud83d\\'

In [146]:
t3 = text_2[4]
t3 = t3[:110] + t3[-10:]
t3
print(json.loads(t3))

# t4 = re.sub(r'[a-zA-Z0-9\\]*\\u[0-9a-fA-F]{4}[a-zA-Z0-9\\]*', '', t3)
t4 = re.sub(r'[.*]*\\u[0-9a-fA-F]{4}[.*]*', '', t3)
print(json.loads(t4))

🎁 FUNNY MOM CANDLE & PETAL BATH BOMB. "Mom' s Last Nerve, Oh look...it's on fire🔥" hu so nice.
 FUNNY MOM CANDLE & PETAL BATH BOMB. "Mom' s Last Nerve, Oh look...it's on fire" hu so nice.


In [145]:
print(t3)
print(t4)

"\ud83c\udf81 FUNNY MOM CANDLE & PETAL BATH BOMB. \"Mom' s Last Nerve, Oh look...it's on fire\ud83d\udd25\" hu so nice."
 FUNNY MOM CANDLE & PETAL BATH BOMB. \"Mom' s Last Nerve, Oh look...it's on  hu so nice."


In [98]:
# for i in range(len(clean_text_2)):
#     print(i)
#     json.loads(clean_text_2[i])

# clean_text_2[4]
json.loads(text_2[4])

'🎁 FUNNY MOM CANDLE & PETAL BATH BOMB. "Mom\' s Last Nerve, Oh look...it\'s on fire🔥" humorous quote can make mom smile every time she looks at them. She will enjoy sleep/yoga/spa/bath time with 100% natural soy wax candle which burns smokelessly & evenly for up to 50 hours. Every time mom pamper herself with anything from this gift set, she\'ll think about who gave her such a thoughtful present. It\'s not just a nice gift, but a reminder that someone cares enough to get her something so nice.'

In [110]:
clean_text_2[4]

'" FUNNY MOM CANDLE & PETAL BATH BOMB. \\"Mom\' s Last Nerve, Oh look...it\'s on " humorous quote can make mom smile every time she looks at them. She will enjoy sleep/yoga/spa/bath time with 100% natural soy wax candle which burns smokelessly & evenly for up to 50 hours. Every time mom pamper herself with anything from this gift set, she\'ll think about who gave her such a thoughtful present. It\'s not just a nice gift, but a reminder that someone cares enough to get her something so nice."'

In [109]:
c3 = clean_text_2[4][:10] + clean_text_2[4][60:]
print(c3)
json.loads(c3)

# clean_text_2

" FUNNY MOh look...it's on " humorous quote can make mom smile every time she looks at them. She will enjoy sleep/yoga/spa/bath time with 100% natural soy wax candle which burns smokelessly & evenly for up to 50 hours. Every time mom pamper herself with anything from this gift set, she'll think about who gave her such a thoughtful present. It's not just a nice gift, but a reminder that someone cares enough to get her something so nice."


JSONDecodeError: Extra data: line 1 column 30 (char 29)

In [97]:
bullets = '\n'.join([f'- {x}' for x in json.loads(text_2[4])])
print(bullets)

- 🎁
-  
- F
- U
- N
- N
- Y
-  
- M
- O
- M
-  
- C
- A
- N
- D
- L
- E
-  
- &
-  
- P
- E
- T
- A
- L
-  
- B
- A
- T
- H
-  
- B
- O
- M
- B
- .
-  
- "
- M
- o
- m
- '
-  
- s
-  
- L
- a
- s
- t
-  
- N
- e
- r
- v
- e
- ,
-  
- O
- h
-  
- l
- o
- o
- k
- .
- .
- .
- i
- t
- '
- s
-  
- o
- n
-  
- f
- i
- r
- e
- 🔥
- "
-  
- h
- u
- m
- o
- r
- o
- u
- s
-  
- q
- u
- o
- t
- e
-  
- c
- a
- n
-  
- m
- a
- k
- e
-  
- m
- o
- m
-  
- s
- m
- i
- l
- e
-  
- e
- v
- e
- r
- y
-  
- t
- i
- m
- e
-  
- s
- h
- e
-  
- l
- o
- o
- k
- s
-  
- a
- t
-  
- t
- h
- e
- m
- .
-  
- S
- h
- e
-  
- w
- i
- l
- l
-  
- e
- n
- j
- o
- y
-  
- s
- l
- e
- e
- p
- /
- y
- o
- g
- a
- /
- s
- p
- a
- /
- b
- a
- t
- h
-  
- t
- i
- m
- e
-  
- w
- i
- t
- h
-  
- 1
- 0
- 0
- %
-  
- n
- a
- t
- u
- r
- a
- l
-  
- s
- o
- y
-  
- w
- a
- x
-  
- c
- a
- n
- d
- l
- e
-  
- w
- h
- i
- c
- h
-  
- b
- u
- r
- n
- s
-  
- s
- m
- o
- k
- e
- l
- e
- s
- s
- l
- y
-  
- &
-  
- e
- v
- e
- n


In [ ]:
# df[df['BULLETS'].str.find('urious, Gentle Clean for Everyday Use - Clearly Herba')>=0]
# print(df.loc[3, 'BULLETS'])
print(df.loc[4332, '_input_text'])

TITLE: Mothers Day & Birthday Gift Baskets for Mom from Daughters Sons Kids, Gift Basket for Grandma Nana Mother in Law Women, Gifts Set for Wife from Husband, [SEP]
ATTRIBUTES:
 [SEP]
FEATURES:
- 🎁 LOOKING FOR GIFTS FOR MOM? This is a great mom gift box for any occasion. Great mothers day gifts for mom. Birthday gifts for mom from daughters, sons, kids and husbands. Customers' top choice when searching for mothers day gift ideas, gift basket for mom, mother-in-law, wife, gifts for mom who has everything unique. It's the best mom gift set for ANYTIME you want to show your appreciation for a trying mother!
- 🎁 WOW HER WITH A CUSTOMIZED MOM GIFT BOX. When she opens it, she will know she's the best mom in the world. Bursting with scent, this mom gift basket includes a 12oz coffee mug with lid and spoon, cozy sock, lavender candle, ocean&rose bath bomb, love knot necklace, jewelry dish, compact mirror, and greeting card. No assembly required. Safe and nice package which can be sent as a Th

In [35]:
print(df.loc[0,'_input_text'])

Title: daVinci Kalani 4-in-1 Convertible Baby Crib - Strong, Easy to Assemble GREENGUARD Certified Crib - Convertible to Toddler Bed, Daybed, Full-Size Bed - Four Adjustable Mattress Heights - Ebony [SEP] Attributes: {"Brand":"DaVinci","Color":"Ebony","Target Audience":"Unisex-Babies","Product Dimensions":"24.5\"L x 13\"W x 51.25\"H","Special Feature":"Sustainable materials"} [SEP] Features: ["Wood","GREENGUARD GOLD CERTIFIED: This product has undergone rigorous scientific testing for over 10,000 chemical emissions and VOCs. It contributes to cleaner indoor air, creating a healthier environment for your baby to sleep, play, and grow","GROWS WITH BABY: Four adjustable mattress positions that can be lowered as your baby begins to sit and stand. Easily converts to toddler bed, daybed, and full-size bed (toddler conversion kit No.M3099 and full-size conversion kit No.M4799 sold separately)","QUALITY MATERIAL: Made of 100% solid sustainable New Zealand pine wood -only the best for your swee

In [ ]:
a = df.loc[0, 'BULLETS']
print()

- Wood
- GREENGUARD GOLD CERTIFIED: This product has undergone rigorous scientific testing for over 10,000 chemical emissions and VOCs. It contributes to cleaner indoor air, creating a healthier environment for your baby to sleep, play, and grow
- GROWS WITH BABY: Four adjustable mattress positions that can be lowered as your baby begins to sit and stand. Easily converts to toddler bed, daybed, and full-size bed (toddler conversion kit No.M3099 and full-size conversion kit No.M4799 sold separately)
- QUALITY MATERIAL: Made of 100% solid sustainable New Zealand pine wood -only the best for your sweet baby
- FOR YOUR BABY'S SAFETY: Say goodbye to toxic chemicals Finished in a non-toxic multi-step painting process and lead and phthalate safe
- COMPLETE THE LOOK: Shop Kalani 3-Drawer Dresser & 6-Drawer Dresser for a coordinated nursery. While this crib is a standard size regular crib, for a best fit we recommend DaVinci's line of non-toxic, GREENGUARD gold mattresses


In [148]:
data_path = 'data/processed/callback_Baby Products.csv'
df = pd.read_csv(data_path, low_memory=False, nrows=20)
tokenizer = AutoTokenizer.from_pretrained('vocab/tokenizer/v1')

def _tokenize(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    return {k: enc[k] for k in ("input_ids", "attention_mask") if k in enc}

ds = Dataset.from_dict({"text": df["_input_text"].tolist()})
ds = ds.map(_tokenize, batched=True, remove_columns=["text"])


Map: 100%|██████████| 20/20 [00:00<00:00, 186.00 examples/s]


In [149]:
from utils.training_funcs import NewTokenForceMaskCollator

def load_new_token_ids(
    tokenizer: AutoTokenizer,
) -> list[int]:
    tokens_path = 'vocab/new_tokens/new_tokens_Baby Products.csv'
    df = pd.read_csv(tokens_path)
    col = df.columns[0]
    tokens = df[col].dropna().str.strip().tolist()
    vocab = tokenizer.get_vocab()
    ids = [vocab[t] for t in tokens if t in vocab]
    print(f"[training_funcs] resolved {len(ids)}/{len(tokens)} new token IDs")
    return ids

new_token_ids = load_new_token_ids(tokenizer)

collator = NewTokenForceMaskCollator(
        tokenizer=tokenizer,
        new_token_ids=new_token_ids,
        mlm_probability=0.15,
    )

model = AutoModelForMaskedLM.from_pretrained('vocab/tokenizer/v1')

[training_funcs] resolved 17/17 new token IDs


Loading weights: 100%|██████████| 137/137 [00:00<00:00, 3364.36it/s]


In [ ]:
loader = DataLoader(
    ds,
    batch_size=4,
    collate_fn=collator,
    shuffle=False,
)
device = 'mps'

_new_token_tensor = torch.tensor(sorted(new_token_ids), dtype=torch.long)
_new_token_tensor = _new_token_tensor.to(device)

model.to(device)

total_loss = 0.0
all_correct = all_total = 0
new_correct = new_total = 0
n_batches = 0
sample_records: list[dict] = []

for batch_idx, batch in enumerate(loader):
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(input_ids=input_ids, labels=labels)
    total_loss += outputs.loss.item()
    n_batches += 1

    preds = outputs.logits.argmax(dim=-1)
    masked = labels != -100

    all_correct += (preds[masked] == labels[masked]).sum().item()
    all_total += masked.sum().item()

    new_mask = masked & torch.isin(labels, _new_token_tensor)
    new_correct += (preds[new_mask] == labels[new_mask]).sum().item()
    new_total += new_mask.sum().item()

    eval_batch_size = 8
    if batch_idx*eval_batch_size < 16:
        print(f'{len(input_ids)=}')
        print(f'{len(labels)=}')
        for i in range(input_ids.shape[0]):
            masked_pos = labels[i] != -100
            label_i = input_ids[i].clone()
            label_i[masked_pos] = labels[i][masked_pos]

            print(f'Callback index = {i}')
            sample_records.append({
                "input": tokenizer.decode(input_ids[i], skip_special_tokens=False, clean_up_tokenization_spaces=False),
                "target": tokenizer.decode(labels[i], clean_up_tokenization_spaces=False),
                "predicted": tokenizer.decode(preds[i], clean_up_tokenization_spaces=False),
            })




len(input_ids)=4
len(labels)=4
Callback index = 0


OverflowError: out of range integral type conversion attempted

In [163]:

idx = 2
print(tokenizer.decode(input_ids[idx], skip_special_tokens=False, clean_up_tokenization_spaces=False))
masked_pos = labels[idx] != -100

print(labels[idx][:50])
label_i = labels[idx]
label_i[~masked_pos] = input_ids[idx][~masked_pos]
print('\n')
print(labels[idx][:50])
print('\n')
print(tokenizer.decode(label_i, skip_special_tokens=False, clean_up_tokenization_spaces=False))

[CLS]TITLE: Artsadd Personalized Baby Blankets[MASK] Boys Girls[MASK] Name Custom[MASK][MASK]urtle Name Blanket for Kids Baby Customized[MASK][MASK] Receiving  precious Baby Gifts for Infant Newborn Birthday[MASK] L [SEP]
ATTRIBUT[MASK]:
Brand : Artsadd
Color : Floral-sea T[MASK]
Style : Casual[MASK]Material : Flee[MASK]
Special Feature : Window Treatments [SEP]
FEATURES:
- 【Personalized Baby Throw Blanket】You can add[MASK][MASK] text[MASK] this cute flower  abandonment.[MASK][MASK] optimize your name into[MASK] funny effect[MASK] This[MASK][MASK] you[MASK] will[MASK] cute.[MASK] click "Customaize Now" on the top right[MASK] start your design and make a cute unique [MASK] for your [MASK] and mom! It's gonna be a special [MASK] just for him/her[MASK]
- 【Custom Gifts forestabl & Kids】Our Personalized Name [MASK] are unique[MASK][MASK]s for [MASK], kids, family, and[MASK] [MASK]s for birth[MASK] and baby showers. It is especially suitable for[MASK][MASK] a swaddling[MASK]blanket for [MASK

In [160]:
idx = 1
print(tokenizer.decode(input_ids[idx], skip_special_tokens=False, clean_up_tokenization_spaces=False))

label_i = labels[idx]
label_i[~masked_pos] = input_ids[idx][~masked_pos]
print('\n')
print(tokenizer.decode(label_i, skip_special_tokens=False, clean_up_tokenization_spaces=False))

[CLS]TITLE: Artsadd Customized Baby [MASK] with Birth Information,[MASK]ized Animal Printed Newborn[MASK]memorate[MASK]et[MASK] Custom Baby [MASK] for Boys Girls for Mom[MASK] Dad, Grandma 30"[MASK]40" [SEP]
ATTRIBUTES dry
Brand : Arts[MASK]
Color : Animal-1[MASK]Style[MASK] Custom
[MASK] : Fleece
Special Feature[MASK] Skin[MASK]ly [SEP]
FEATURES:
- 【Custom preachingborn Baby [MASK][MASK]�[MASK] miss the moment of [MASK] born! Click[MASK]Customize Now”, update your [MASK]'s name, birth date, weight, length, and birth time. To get a unique [MASK] birth memorial [MASK]. The personalized [MASK] provide[MASK] cute cartoon[MASK] for you choose. You[MASK] use it as[MASK][MASK] take pictures of your [MASK][MASK]s precious moments. And share your photos with familystem friends!
-  【The Perfect Size & Material �[MASK]This custom [MASK] [MASK] is made from 100% high quality ultra-soft Micro Fleece,[MASK], skin[MASK]friendly and [MASK]. The customize [MASK][MASK][MASK] sizes for the  Reddit girls

OverflowError: out of range integral type conversion attempted

In [162]:
labels[idx][:30]

tensor([50281,    53, 43561,  -100,  -100,  1911, 12047,  1025,  -100,   209,
        50284,   342,  -100,  -100,    13, 23307,  -100, 18630,  -100,   264,
         1457,  6448, 50284,  6441,  -100, 48517,   292, 50284, 12047, 25707],
       device='mps:0')

In [161]:
label_i

tensor([50281,    53, 43561,  -100,  -100,  1911, 12047,  1025,  -100,   209,
        50284,   342,  -100,  -100,    13, 23307,  -100, 18630,  -100,   264,
         1457,  6448, 50284,  6441,  -100, 48517,   292, 50284, 12047, 25707,
          209, 50284,   323, 27014, 26152,   323, 13541, 50284, 16522,    13,
         8481,   785,  1884,     3,    89,  1449,     3,  -100, 50282,   187,
        21840, 30639,  1410,  6079,  -100, 49600,  -100, 15118, 50284,   187,
         -100,  -100, 18630,  -100,    18, 50284,  -100, 50284, 12047,   187,
        50284,  1163,   401, 14906,  -100,   187, 22166, 36350, 50284, 37019,
        50284,   314,   209, 50282,   187, 12297,  -100, 43087,  -100,   187,
           14,   209,  2155,   227, 13510, 43560,  6448, 25707,   209, 50284,
        50284,   228, 15751,  -100,   253,  2774,   273,  -100, 50284,  5686,
            2, 15682, 50284, 13510,   907,  3954,  -100,  5731,   634,  -100,
        50284,  -100,  -100,    13,  4201,  3522,    13,  2801, 

In [23]:
# tokenizer.decode(labels[i], clean_up_tokenization_spaces=False)
# labels[i].shape

# tokenizer.decode([1,2,3,4,5,6,-100], clean_up_tokenization_spaces=False)
tokenizer.decode(preds[i], clean_up_tokenization_spaces=False)

'[CLS]Title: daVinci Kalani 4-in-1 Convertible Baby Crib - Strong and Easy to Assemble GREENGUARD Certified Crib - Convertible to \xa0 Bed, Daybed and Full-Size Bed - Four Adjustable Mattress Heights - Ebony ### Attributes: {"Brand":"DaVinci","Color":"Ebony","Target Audience":"Fids-Babies","Product Dimensions":"8.5\\"L x 13\\"W x 51.25\\"H","Special Feature":"Sustainable materials"} \nTitle: ["Wood","GREENGUARD GOLD CERTIFIED: This Inf has undergone rigorous scientific testing of over 10,000 chemical compounds and VOCs. It contributes to cleaner indoor air, and a healthier environment for your ick to sleep, play, and grow","GROWS WITH BABY: Four adjustable mattress positions that can be lowered as your ia begins to sit and stand. Easily converts to \xa0 bed, daybed, and full-size bed (day conversion kit No.M3099 and full-size conversion kit No.M4799 sold separately)","QUALITY MATERIAL: Made of 100% solid, New Zealand pine wood -only the best for your sweet ia","FOR THE BABY\'S SAFETY: 

In [25]:
label_i = input_ids[i]
label_i[masked_pos] = labels[i][masked_pos]

a = tokenizer.decode(input_ids[i], skip_special_tokens=False, clean_up_tokenization_spaces=False)
b = tokenizer.decode(label_i, clean_up_tokenization_spaces=False)
print(a)
print(b)
a == b

[CLS]Title: daVinci Kalani 4-in-1 Convertible Baby Crib - Strong, Easy to Assemble GREENGUARD Certified Crib - Convertible to Toddler Bed, Daybed, Full-Size Bed - Four Adjustable Mattress Heights - Ebony [SEP] Attributes: {"Brand":"DaVinci","Color":"Ebony","Target Audience":"Unisex-Babies","Product Dimensions":"24.5\"L x 13\"W x 51.25\"H","Special Feature":"Sustainable materials"} [SEP] Features: ["Wood","GREENGUARD GOLD CERTIFIED: This product has undergone rigorous scientific testing for over 10,000 chemical emissions and VOCs. It contributes to cleaner indoor air, creating a healthier environment for your baby to sleep, play, and grow","GROWS WITH BABY: Four adjustable mattress positions that can be lowered as your baby begins to sit and stand. Easily converts to toddler bed, daybed, and full-size bed (toddler conversion kit No.M3099 and full-size conversion kit No.M4799 sold separately)","QUALITY MATERIAL: Made of 100% solid sustainable New Zealand pine wood -only the best for your

True

In [30]:
tokenizer.decode(input_ids[i][masked_pos], skip_special_tokens=False, clean_up_tokenization_spaces=False)

# masked_pos

' Baby,Toddlerbed,ress:Brand AudUnisex24W x 51 FeaturesERT product has undergone for over emissions V. It creatingbaby and: Fourbaby begins standastoddlertoddler conversion."," Made sustainablebaby YOURYSAF goodbye a andPLEOOK & 6Draw nursery c regular c a we Da\'s lineGU'